# NYCDOE Office of Data Management Interview Data Skills Assessment Task

# Le' Sean Robrts (CUNY CITY TECH Graduate), Prospect Data Analysts at NYCDOE ODM

# April 16th, 2026

[Emai](lesean.roberts85@gmail.com)

[LinkedIN](https://www.linkedin.com/in/le-sean-roberts)

[GitHub](https://github.com/LeSeanRoberts)

## Part I: Foundational Definitions (NYSED Standards)

Concerning the Office of Data Management (ODM) of the New York City Department of Education (NYCDOE), the following result is the completion of properly identified specific definitions or concepts from the New York State Education Department (NYSED) for intended exhibition by the Diversity Equity and Inclusion (DEI) Office:

**4-Year Cohort:** Such identifies the group of students who first entered 9th grade in a specific school year (e.g., the 2017 cohort entered in 2017-18). This grouping is updated annually to include students who transfer in and exclude those who transfer out, emigrate, or pass away (NYSED, n.d.-c).

**Graduate:** Students who receive an accredited academic degree or diploma. A student within the cohort who earned a regular high school diploma (Regents, Advanced Regents, or Local) by the end of their fourth year. For reporting purposes, NYC usually highlights "August" graduates to include students who finished over the summer.

**Dropout:** A student who left (high) school without earning a diploma or a (high) school equivalency (HSE) and did not transfer to another diploma-granting program or educational setting. Under NYSED attendance reporting guidelines, a student who is absent without excuse for 20 consecutive days is considered to have ceased attendance and is classified as a "dropout" unless school officials have substantiated that the student has transferred to another educational program or cannot attend due to illness/injury.

**Completer:** Students who depart from their enrollment with a Career Development and Occupational Studies (CDOS) commencement credential, or a Skills & Achievement commencement credential as a alternative to local or Regents diploma. Completers are not counted as graduates; they are "completers" who have met specific alternative requirements. A High School Equivalency (HSE) diploma, such as the TASC/GED applies as well.

**Graduation Rate**: A percentage calculated by dividing the number of graduates in the adjusted cohort (often within four years) by the total number of students in that cohort.

## Part II: Demographics for Graduation Rate

This following analysis was developed to support the NYCDOE DEI Office’s federal grant application, which requires a transparent presentation of disparities in graduation outcomes across school districts. Applying the 2017 cohort 4-year August graduation data, the purpose of this review or analysis is to validate the accuracy of key reported metrics and confirm that subgroup differences are correctly represented. The focus is on citywide graduation rates and variations across gender, race/ethnicity, English language learner (ELL) status, and disability status.

### Primal Phase 

The process begins with data assimilation involving the extraction of multiple spreadsheets within a single document. Following, a glimpse of the data structure to identify any possible need of data wrangling tasks before computation with any tabular structures (like dataframes). 

In [2]:
import pandas as pd

# Loading the files
all_df = pd.read_excel('2021-graduation_rates_public_district candidate.xlsx',
                       sheet_name='All')
gender_df = pd.read_excel('2021-graduation_rates_public_district candidate.xlsx',
                          sheet_name='Gender')
ethnicity_df = pd.read_excel('2021-graduation_rates_public_district candidate.xlsx',
                             sheet_name='Ethnicity')
ell_df = pd.read_excel('2021-graduation_rates_public_district candidate.xlsx',
                       sheet_name='ELL')
swd_df = pd.read_excel('2021-graduation_rates_public_district candidate.xlsx',
                       sheet_name='SWD')
dist_info_df = pd.read_excel('2021-graduation_rates_public_district candidate.xlsx',
                             sheet_name='District Info')

Data Introspection:

In [4]:
import io

def section(title, df, show_unique=False, show_info=False):
    print("\n" + "="*70)
    print(f"--- {title} ---")
    print("="*70)

    # Head
    print("\n[HEAD]")
    print(df.head())

    # Info (captured cleanly)
    if show_info:
        print("\n[INFO]")
        buffer = io.StringIO()
        df.info(buf=buffer)
        print(buffer.getvalue())

    # Unique districts
    if show_unique and 'District' in df.columns:
        print("\n[UNIQUE DISTRICTS]")
        print(df['District'].unique())

    print("\n" * 2)



section("All STUDENTS", all_df, show_unique=True, show_info=True)
section("GENDER", gender_df, show_unique=True, show_info=True)
section("ETHNICITY", ethnicity_df, show_unique=True, show_info=True)
section("ELL", ell_df, show_info=True)
section("SWD", swd_df, show_info=True)
section("DISTRICT INFO", dist_info_df, show_info=True)


--- All STUDENTS ---

[HEAD]
   District      Category  Cohort Year         Cohort  # Total Cohort  \
0         2  All Students         2017  4 year August            8852   
1         1  All Students         2016  4 year August            1047   
2         1  All Students         2015  4 year August            1006   
3         1  All Students         2014  4 year August             957   
4         1  All Students         2013  4 year August            1043   

   # Grads    % Grads  # Total Regents  % Total Regents of Cohort  \
0     7325  82.749664             7265                  82.071846   
1      713  68.099335              687                  65.616043   
2      689  68.489067              641                  63.717693   
3      631  65.935211              597                  62.382446   
4      639  61.265579              608                  58.293385   

   % Total Regents of Grads  ...  % Local of Cohort  % Local of Grads  \
0                 99.180885  ...           

### Secondary Phase

This analysis processes graduation data for the 2017 4-year August cohort by cleaning non-numeric values (like 's', often used for data suppression) and aggregating totals across different demographic dimensions.

**Key Analytical Observations**

1. Data Integrity: By converting # Total Cohort and # Grads to numeric values with errors='coerce', the script effectively handles missing or suppressed data. This ensures that the sums reflect only valid, reported figures.

2. Graduation Rate Calculation: The formula applied --

$$\% \text{ Grad Rate} = \left( \frac{\sum \# \text{ Grads}}{\sum \# \text{ Total Cohort}} \right) \times 100$$

This provides a weighted average for each category, preventing smaller sub-groups from disproportionately skewing the citywide percentage.

3. Demographic Comparison: By running the function across five distinct DataFrames (all, gender, ethnicity, ell, and swd), you can identify performance gaps. For example:

       Ethnicity: Comparing graduation rates between different racial groups to identify equity gaps.

       ELL & SWD: Assessing the outcomes for English Language Learners and Students with Disabilities relative to the "All Students" baseline.

       Gender: Identifying variations in completion rates between male and female students.

The output prints five tables, each showing the Volume (Total Cohort), the Success Count (Grads), and the Success Percentage (Grad Rate). This structure allows for a quick comparison of which student populations are meeting graduation milestones and where additional support may be required.

In [6]:
import numpy as np

def clean_and_sum(df, cohort_year=2017, cohort_type='4 year August'):
    # Filter for the specific cohort
    subset = df[(df['Cohort Year'] == cohort_year) & (df['Cohort'] == cohort_type)].copy()
    
    # Replace 's' with NaN
    subset['# Total Cohort'] = pd.to_numeric(subset['# Total Cohort'], errors='coerce')
    subset['# Grads'] = pd.to_numeric(subset['# Grads'], errors='coerce')
    
    # Group by Category and sum
    summary = subset.groupby('Category').agg({
        '# Total Cohort': 'sum',
        '# Grads': 'sum'
    }).reset_index()
    
    summary['% Grad Rate'] = (summary['# Grads'] / summary['# Total Cohort']) * 100
    return summary

# Citywide Calculations
citywide_all = clean_and_sum(all_df)
citywide_gender = clean_and_sum(gender_df)
citywide_ethnicity = clean_and_sum(ethnicity_df)
citywide_ell = clean_and_sum(ell_df)
citywide_swd = clean_and_sum(swd_df)

print("Citywide All Students:")
print(citywide_all)
print("\nCitywide Gender:")
print(citywide_gender)
print("\nCitywide Ethnicity:")
print(citywide_ethnicity)
print("\nCitywide ELL:")
print(citywide_ell)
print("\nCitywide SWD:")
print(citywide_swd)

Citywide All Students:
       Category  # Total Cohort  # Grads  % Grad Rate
0  All Students           74738    60689    81.202333

Citywide Gender:
  Category  # Total Cohort  # Grads  % Grad Rate
0   Female           35967    30973    86.115050
1     Male           38771    29716    76.644915

Citywide Ethnicity:
          Category  # Total Cohort  # Grads  % Grad Rate
0            Asian           13031  11838.0    90.844908
1            Black           18026  14154.0    78.519916
2         Hispanic           29087  22743.0    78.189569
3     Multi-Racial            1385   1092.0    78.844765
4  Native American             853    672.0    78.780774
5            White           12356  10104.0    81.774037

Citywide ELL:
     Category  # Total Cohort  # Grads  % Grad Rate
0         ELL            8376     5052    60.315186
1  Former ELL            4198     3880    92.424964
2     Not ELL           62164    51757    83.258799

Citywide SWD:
  Category  # Total Cohort  # Grads  % Grad Ra

### Analysis

1. **The Gender Gap**

There is a notable 9.47% performance gap between female and male students.

Females: 86.1% graduation rate.

Males: 76.6% graduation rate.
Despite having a larger cohort size (38,771), male students had fewer total graduates (29,716) than their female counterparts.

2. **Ethnicity and Performance**

Graduation rates vary significantly across ethnic categories, with a 12.66% spread between the highest and lowest-performing groups.

High Performers: Asian students lead the city with a 90.8% rate.

Mid-Range: White students follow at 81.8%.

Lower-Range: Black (78.5%), Multi-Racial (78.8%), Native American (78.8%), and Hispanic (78.2%) groups show very similar graduation levels, trailing the citywide average.

3. **Specialized Student Populations (ELL & SWD)**

The most drastic disparities appear in the English Language Learner (ELL) and Students with Disabilities (SWD) categories.

The "Former ELL" Boost: Interestingly, students who transitioned out of ELL status (Former ELL) have the highest graduation rate in the entire dataset at 92.4%, surpassing even the Asian demographic and "Not ELL" students.

Systemic Challenges: Current ELL students (60.3%) and SWD students (57.8%) graduate at rates significantly lower than the citywide average of 81.2%. The 29.96% gap between SWD and Not-SWD students represents the largest performance disparity in the data.


## Part III: Equity Gaps in Graduation Rates Across Districts 20, 22, 25 & 27

Developded analysis for a localized geographic and demographic drill-down, particularly focusing on four school districts (20, 22, 25, 27). This approach is designed to uncover how socio-economic factors and ethnicity are disbursed involving graduation outcomes at a community level.

1. **The Socio-Economic Link (ENI)**

By merging the graduation data with dist_info_df, the code introduces the Economic Needs Index (ENI).

Purpose: The ENI estimates the economic hardship of a student population based on factors like poverty and temporary housing.

Analysis: This allows you to correlate graduation performance with resource access. Typically, districts with a lower ENI (wealthier) show higher graduation rates, while those with higher ENI scores may require more targeted funding to overcome systemic barriers.

2. **Comparative District Performance**

The initial printout provides a snapshot of how these specific districts compare to one another:

Volume vs. Success: It shows the scale of the student body (# Total Cohort) alongside the success rate (% Grads).

Borough Context: It identifies where these districts sit (e.g., Brooklyn or Queens), helping to account for regional differences in school infrastructure or local policy.

3. **Ethnicity Disparity Pivot**

The second half of the code creates a Pivot Table, which is the most effective way to spot achievement gaps.

**Identifying Gaps:** By aligning ethnic categories side-by-side for each district, there's ability to observe if a specific group is being underserved or is underperforming in one district compared to another.

**Suppression Awareness:** The use of pd.to_numeric(..., errors='coerce') is critical here. In smaller districts, certain ethnic subgroups may have their data "suppressed" (shown as 's') to protect student privacy if the group size is very small. These will appear as NaN in your pivot table, preventing skewed or misleading averages.

There is transition from from descriptive (what happened citywide) to diagnostic (why certain areas or groups might be performing differently). It sets the stage for data-driven decisions on where to allocate supplemental educational resources.

In [7]:
# Part III: Districts 20, 22, 25, 27
target_districts = [20, 22, 25, 27]
dist_summary = all_df[(all_df['Cohort Year'] == 2017) & 
                      (all_df['Cohort'] == '4 year August') & 
                      (all_df['District'].isin(target_districts))].copy()

# Add demographic info
dist_summary = dist_summary.merge(dist_info_df, left_on='District', right_on='district')

print("\n--- Target Districts Analysis ---")
print(dist_summary[['District', 'borough', '# Total Cohort', '# Grads', '% Grads', 'avg school Economic Needs Index (ENI)']])

# Check for disparities by subgroup in these districts
# Let's check Ethnicity in these districts
target_ethnicity = ethnicity_df[(ethnicity_df['Cohort Year'] == 2017) & 
                                (ethnicity_df['Cohort'] == '4 year August') & 
                                (ethnicity_df['District'].isin(target_districts))].copy()

# Print target ethnicity grad rates
print("\nEthnicity Grad Rates in Target Districts:")
target_ethnicity['% Grads'] = pd.to_numeric(target_ethnicity['% Grads'], errors='coerce')
pivot_ethnicity = target_ethnicity.pivot(index='District', columns='Category', values='% Grads')
print(pivot_ethnicity)


--- Target Districts Analysis ---
   District   borough  # Total Cohort  # Grads    % Grads  \
0        20  Brooklyn            3534     2588  73.231468   
1        22  Brooklyn            2963     2487  83.935204   
2        27    Queens            2924     2330  79.685364   
3        25    Queens            2530     1985  78.458496   

   avg school Economic Needs Index (ENI)  
0                                   0.75  
1                                   0.65  
2                                   0.70  
3                                   0.62  

Ethnicity Grad Rates in Target Districts:
Category      Asian      Black   Hispanic  Multi-Racial  Native American  \
District                                                                   
20        83.629890  76.397514  67.022308     49.090908        54.545456   
22        96.374626  85.093170  80.620155     72.222221        92.857140   
25        86.576355  81.879196  72.026642     81.538460        80.769234   
27        80.758430  

### Analysis for Districts 20, 22, 25, and 27

The analysis of Districts 20, 22, 25, and 27 reveals significant localized variations in performance, particularly when correlating economic need with graduation outcomes across ethnic subgroups.

1. **Economic Correlation (ENI)**

The Economic Needs Index (ENI) serves as a strong predictor of district-wide success in this subset.

District 22 has the lowest economic need (0.65) and the highest overall graduation rate (83.9%).

District 20 has the highest economic need (0.75) and the lowest overall graduation rate (73.2%).

This suggests that for every 10% increase in the economic needs index, there is a roughly corresponding 10% drop in the graduation rate within these specific districts.

2. **Ethnicity Performance Gaps**

The pivot table identifies several demographic outliers that differ from citywide trends:

Asian Students: Lead in three out of four districts, peaking at a near-perfect 96.4% in District 22.

District 20 Challenges: This district shows the sharpest internal disparities. While Asian students graduate at 83.6%, Multi-Racial and Native American students in the same district fall significantly lower (49.1% and 54.5%, respectively).

Hispanic Performance: District 27 is the only one in the group where Hispanic students (81.1%) outperform the district's Asian students (80.8%) and White students (78.0%).

3. **Subgroup Stability**

White Students: Interestingly, White student graduation rates remain relatively flat across these districts, fluctuating in a narrow band between 69.5% and 78.0%, showing less volatility than other groups regardless of the district's ENI.

Black Students: Show high consistency in these target areas, maintaining a graduation rate between 76.4% and 85.1%, often outperforming the citywide Hispanic and Black averages.

**Summary Table:**

| District | Top Performing Group | Rate | Lowest Performing Group | Rate | 
|----------|----------------------|------|-------------------------|------|
| 20 | Asian | 83.6% | Multi-Racial | 49.1% |
| 22 | Asian | 96.4% | Multi-Racial | 72.2% | 
| 25 | Asian | 86.6% | Hispanic | 72.0% | 
| 27 | Hispanic | 81.1% | Native American | 73.8% |

**NOTE:** one should keep in mind that proportions can lead to a poor picture. Namely, comparing distinct groups with disproportionate populations. For example, if one group has 3 individuals in total, while the other group has 10 individuals in total. If only one student in each group fails to successfully matriculate...it's a 67% success rate versus a 90% success rate. 

### Analysis for "Completer" Success in Such Districts:

In [8]:
# Calculate Completer Rate
def calculate_completer_rate(df, cohort_year=2017, cohort_type='4 year August'):
    subset = df[(df['Cohort Year'] == cohort_year) & (df['Cohort'] == cohort_type)].copy()
    subset['# Total Cohort'] = pd.to_numeric(subset['# Total Cohort'], errors='coerce')
    subset['# Grads'] = pd.to_numeric(subset['# Grads'], errors='coerce')
    subset['# TASC (GED)'] = pd.to_numeric(subset['# TASC (GED)'], errors='coerce')
    
    summary = subset.groupby('Category').agg({
        '# Total Cohort': 'sum',
        '# Grads': 'sum',
        '# TASC (GED)': 'sum'
    }).reset_index()
    
    summary['# Completers'] = summary['# Grads'] + summary['# TASC (GED)']
    summary['% Completer Rate'] = (summary['# Completers'] / summary['# Total Cohort']) * 100
    summary['% Grad Rate'] = (summary['# Grads'] / summary['# Total Cohort']) * 100
    return summary

cw_all = calculate_completer_rate(all_df)
cw_gender = calculate_completer_rate(gender_df)
cw_ethnicity = calculate_completer_rate(ethnicity_df)
cw_ell = calculate_completer_rate(ell_df)
cw_swd = calculate_completer_rate(swd_df)

# Consolidate citywide summary for the memo
citywide_summary = pd.concat([cw_all, cw_gender, cw_ethnicity, cw_ell, cw_swd], axis=0)
citywide_summary.to_csv('citywide_graduation_analysis.csv', index=False)

# Specific District Analysis for Memo
dist_detail = all_df[(all_df['Cohort Year'] == 2017) & 
                     (all_df['Cohort'] == '4 year August') & 
                     (all_df['District'].isin([20, 22, 25, 27]))].copy()
dist_detail = dist_detail.merge(dist_info_df, left_on='District', right_on='district')
dist_detail = dist_detail[['District', 'borough', '# Total Cohort', '# Grads', '% Grads', 'avg school Economic Needs Index (ENI)', '% black students', '% hispanic students']]
dist_detail.to_csv('target_district_analysis.csv', index=False)

print(citywide_summary)
print(dist_detail)

          Category  # Total Cohort  # Grads  # TASC (GED)  # Completers  \
0     All Students           74738  60689.0         566.0       61255.0   
0           Female           35967  30973.0         210.0       31183.0   
1             Male           38771  29716.0         356.0       30072.0   
0            Asian           13031  11838.0          54.0       11892.0   
1            Black           18026  14154.0         178.0       14332.0   
2         Hispanic           29087  22743.0         275.0       23018.0   
3     Multi-Racial            1385   1092.0           2.0        1094.0   
4  Native American             853    672.0          13.0         685.0   
5            White           12356  10104.0          44.0       10148.0   
0              ELL            8376   5052.0         105.0        5157.0   
1       Former ELL            4198   3880.0          21.0        3901.0   
2          Not ELL           62164  51757.0         440.0       52197.0   
0          Not SWD       

This analysis expands the scope of educational success by introducing the Completer Rate, which accounts for students earning a high school equivalency (TASC/GED or other) in addition to traditional diplomas. It also provides a localized look at how district demographics relate to overall achievement.

1. **Grad Rate vs. Completer Rate**

The "Completer Rate" provides a more inclusive metric of student success. Citywide, the TASC/GED adds 566 students to the success count, raising the overall rate from 81.2% to 82.0%.

Gender Gap: Male students utilized the TASC pathway more frequently than females (356 vs. 210), though females still maintain a significant lead in the total completer rate (86.7% vs 77.6%).

SWD & ELL Impact: These populations see the largest relative "bump" from the TASC pathway. For Students with Disabilities (SWD), the success rate increases by 1.16% when including equivalency diplomas.

2. **District Demographics and Performance**

The target district analysis reveals how specific demographic concentrations and economic factors influence localized outcomes.

Economic Sensitivity: District 20, despite having the lowest percentage of Black students (1.3%), has the lowest graduation rate (73.2%) in this group. This aligns with its high Economic Needs Index (0.75), suggesting that economic hardship is a primary driver of the performance gap here.

High Performance in Diverse Districts: District 22 stands out with the highest graduation rate (83.9%) despite a high percentage of Black students (41.8%), likely supported by a lower ENI (0.65) compared to District 20.

3. **Key Observations**

The Equivalency Buffer: While traditional graduation is the goal, the TASC/GED is a critical safety net for nearly 1,000 students citywide, particularly for males and SWD populations.

Beyond Demographics: The data suggests that Economic Need (ENI) is often a more aggressive predictor of district-wide graduation rates than specific racial compositions. District 25 and 22, which have the lowest ENIs, consistently outperform District 20 and 27.


## References

NYSED (n.d.-c). What is the Difference Between the 4-, 5-, and 6-Year Graduation Rates? – New York State Education Department. https://datasupport.nysed.gov/hc/en-us/articles/26044709568013-What-is-the-difference-between-the-4-5-and-6-year-graduation-rates 